# Hyperparameter Tuning and Threshold Optimization

This notebook demonstrates how to:
1. Create train/validation/test splits
2. Tune Random Forest hyperparameters
3. Tune XGBoost hyperparameters
4. Optimize prediction thresholds
5. Evaluate final model on test set

In [2]:
# automatically reload imported modules before executing code
%load_ext autoreload
%autoreload 2

In [3]:
from pyrekordbox import Rekordbox6Database
import polars as pl
from nbutils import setup_path

setup_path()
db = Rekordbox6Database()

pl.Config.set_tbl_rows(30)

[23:05:03] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


polars.config.Config

## 1. Prepare Data with Validation Split

In [4]:
from utils import get_base_dataset
from processing import preprocess_tag_group

# Get base dataset
base_df = get_base_dataset(db, min_tag_count=5)

# Load features
features_df = pl.read_parquet("../data/song_features.parquet")

# Define feature columns to exclude
exclude_cols = [
    "song_path",
    "harmonic_percussive_ratio",
    "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]

feature_cols = [col for col in features_df.columns 
                if col not in exclude_cols + ["song_id", "song_path"]]

# Filter features
features_df = features_df.filter(pl.col("energy_increase_ratio").is_not_null())

# Preprocess with validation split
# test_size=0.2 means 20% test set
# val_size=0.2 means 20% of remaining data (so 0.2 * 0.8 = 16% of total) for validation
# Final split: 64% train, 16% val, 20% test
result = preprocess_tag_group(
    base_df, features_df, "Genre",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,  # NEW: Create validation set
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

print(f"\nData split:")
print(f"  Train: {result.X_train.shape[0]} samples")
print(f"  Validation: {result.X_val.shape[0]} samples")
print(f"  Test: {result.X_test.shape[0]} samples")
print(f"  Total labels: {len(result.tags)}")


Filtering tags with fewer than 5 occurrences:

Genre:
  - Grime: 1 occurrence(s)
  - New Beat: 3 occurrence(s)
  - Dancehall: 3 occurrence(s)
  - Gabber: 4 occurrence(s)
  - Blues: 4 occurrence(s)

Mood:
  - Industrial: 1 occurrence(s)

Total tags filtered: 6


PREPROCESSING: GENRE

Preparing multi-label data for 'Genre':
Unique songs: 696
Unique tags: 43
Tags: Acid, Afrobeat, Ambient, Ballad, Bass, Beats, Boogie, Breakbeat, Disco, Drum & Bass, Dubstep, Electro, Eurodance, Folk, World & Country, Footwork, Funk, Future Bass, Garage, Hip-Hop, House, Indie, Jazz, Juke, Jungle, Motown, Neo Soul, New Wave, Old School, Pop, Punk, R&B, Rap, Reggae, Reggaeton, Rock, Rock & Roll, Singer-Songwriter, Soul, Techno, Trance, Tribal, Trip-Hop, Yaught Rock

Final dataset shape:
  X: (499, 361) (song_id + 360 features)
  y: (499, 43) (43 binary labels)
  Average tags per song: 3.48


Genre - Initial label statistics:
shape: (43, 3)
┌───────────────────┬───────┬────────────┐
│ tag               ┆ count

## 2. Random Forest Hyperparameter Tuning

We'll use random search first for quick exploration, then grid search on a narrower range.

In [6]:
from models import tune_random_forest_hyperparams

# Random search - quick exploration
rf_results = tune_random_forest_hyperparams(
    X_train=result.X_train,
    y_train=result.y_train,
    X_val=result.X_val,
    y_val=result.y_val,
    tags=result.tags,
    search_type='random',
    n_iter=30,  # Test 30 random configurations
    verbose=True
)

print("\nTop 5 configurations:")
print(rf_results['all_results'].head(5))


RANDOM FOREST HYPERPARAMETER TUNING
Search type: random

Testing 30 random parameter combinations...

Progress: 5/30 combinations tested
Progress: 10/30 combinations tested
Progress: 15/30 combinations tested
Progress: 20/30 combinations tested
Progress: 25/30 combinations tested
Progress: 30/30 combinations tested

TUNING RESULTS

Best macro F1 score: 0.2977

Best parameters:
  n_estimators: 100
  max_depth: None
  min_samples_split: 5
  min_samples_leaf: 10
  max_features: None
  class_weight: balanced
  max_samples: 0.8



Top 5 configurations:
shape: (5, 8)
┌────────────┬───────────┬────────────┬────────────┬────────────┬───────────┬───────────┬──────────┐
│ n_estimato ┆ max_depth ┆ min_sample ┆ min_sample ┆ max_featur ┆ class_wei ┆ max_sampl ┆ macro_f1 │
│ rs         ┆ ---       ┆ s_split    ┆ s_leaf     ┆ es         ┆ ght       ┆ es        ┆ ---      │
│ ---        ┆ i64       ┆ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ f64      │
│ i64        ┆           ┆ 

## 3. XGBoost Hyperparameter Tuning

In [11]:
from models import tune_xgboost_hyperparams

# Random search for XGBoost
xgb_results = tune_xgboost_hyperparams(
    X_train=result.X_train,
    y_train=result.y_train,
    X_val=result.X_val,
    y_val=result.y_val,
    tags=result.tags,
    search_type='random',
    n_iter=100,
    verbose=True
)

print("\nTop 5 configurations:")
print(xgb_results['all_results'].head(5))


XGBOOST HYPERPARAMETER TUNING
Search type: random

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best macro F1 score: 0.3718

Best parameters:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.1
  subsample: 1.0
  colsample_bytree: 0.6
  

## 4. Threshold Optimization

Now optimize prediction thresholds on the validation set for **both** models.
This is especially important for XGBoost which tends to be too conservative with default thresholds.

In [12]:
from models import optimize_prediction_thresholds

# Optimize thresholds for Random Forest
print("\n" + "="*70)
print("RANDOM FOREST - THRESHOLD OPTIMIZATION")
print("="*70)
rf_threshold_results = optimize_prediction_thresholds(
    model=rf_results['best_model'],
    X_val=result.X_val,
    y_val=result.y_val,
    tags=result.tags,
    metric='f1',
    verbose=True
)

# Optimize thresholds for XGBoost
print("\n" + "="*70)
print("XGBOOST - THRESHOLD OPTIMIZATION")
print("="*70)
xgb_threshold_results = optimize_prediction_thresholds(
    model=xgb_results['best_model'],
    X_val=result.X_val,
    y_val=result.y_val,
    tags=result.tags,
    metric='f1',
    verbose=True
)

# Compare improvements
print("\n" + "="*70)
print("THRESHOLD OPTIMIZATION COMPARISON")
print("="*70)
print(f"Random Forest improvement: {rf_threshold_results['improvement']:+.4f}")
print(f"XGBoost improvement: {xgb_threshold_results['improvement']:+.4f}")
print(f"\nRandom Forest with thresholds: {rf_threshold_results['optimized_metrics']['macro_f1']:.4f}")
print(f"XGBoost with thresholds: {xgb_threshold_results['optimized_metrics']['macro_f1']:.4f}")

# Choose the best overall model
if xgb_threshold_results['optimized_metrics']['macro_f1'] > rf_threshold_results['optimized_metrics']['macro_f1']:
    best_model = xgb_results['best_model']
    best_thresholds = xgb_threshold_results['thresholds']
    best_model_name = "XGBoost"
    threshold_results = xgb_threshold_results
else:
    best_model = rf_results['best_model']
    best_thresholds = rf_threshold_results['thresholds']
    best_model_name = "Random Forest"
    threshold_results = rf_threshold_results

print(f"\nBest model: {best_model_name}")


RANDOM FOREST - THRESHOLD OPTIMIZATION

THRESHOLD OPTIMIZATION
Optimizing for: f1



/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Default macro F1: 0.2977
Optimized macro F1: 0.4909
Improvement: +0.1931

Per-label thresholds (sorted by F1 improvement):
shape: (22, 5)
┌────────────┬───────────┬────────────┬──────────────┬─────────────┐
│ tag        ┆ threshold ┆ default_f1 ┆ optimized_f1 ┆ improvement │
│ ---        ┆ ---       ┆ ---        ┆ ---          ┆ ---         │
│ str        ┆ f64       ┆ f64        ┆ f64          ┆ f64         │
╞════════════╪═══════════╪════════════╪══════════════╪═════════════╡
│ Soul       ┆ 0.311597  ┆ 0.0        ┆ 0.748299     ┆ 0.748299    │
│ Rock       ┆ 0.399839  ┆ 0.0        ┆ 0.533333     ┆ 0.533333    │
│ House      ┆ 0.381386  ┆ 0.333333   ┆ 0.819048     ┆ 0.485714    │
│ Rap        ┆ 0.378988  ┆ 0.1        ┆ 0.571429     ┆ 0.471429    │
│ Trance     ┆ 0.292673  ┆ 0.0        ┆ 0.451613     ┆ 0.451613    │
│ Techno     ┆ 0.374152  ┆ 0.0        ┆ 0.428571     ┆ 0.428571    │
│ Old School ┆ 0.361178  ┆ 0.25       ┆ 0.666667     ┆ 0.416667    │
│ Tribal     ┆ 0.291852  ┆ 0.0    

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## 5. Final Evaluation on Test Set

Now evaluate the tuned model with optimized thresholds on the held-out test set.

In [13]:
# Compare threshold distributions
comparison = pl.DataFrame({
    "tag": result.tags,
    "rf_threshold": rf_threshold_results['thresholds'],
    "xgb_threshold": xgb_threshold_results['thresholds'],
    "threshold_diff": rf_threshold_results['thresholds'] - xgb_threshold_results['thresholds'],
    "rf_f1_improvement": rf_threshold_results['threshold_df']['improvement'],
    "xgb_f1_improvement": xgb_threshold_results['threshold_df']['improvement'],
}).sort("threshold_diff", descending=True)

print("\nThreshold comparison (sorted by difference):")
print(comparison)

print(f"\nRandom Forest threshold range: [{rf_threshold_results['thresholds'].min():.3f}, {rf_threshold_results['thresholds'].max():.3f}]")
print(f"XGBoost threshold range: [{xgb_threshold_results['thresholds'].min():.3f}, {xgb_threshold_results['thresholds'].max():.3f}]")
print(f"\nRF avg threshold: {rf_threshold_results['thresholds'].mean():.3f}")
print(f"XGB avg threshold: {xgb_threshold_results['thresholds'].mean():.3f}")


Threshold comparison (sorted by difference):
shape: (22, 6)
┌────────────┬──────────────┬───────────────┬────────────────┬──────────────────┬──────────────────┐
│ tag        ┆ rf_threshold ┆ xgb_threshold ┆ threshold_diff ┆ rf_f1_improvemen ┆ xgb_f1_improveme │
│ ---        ┆ ---          ┆ ---           ┆ ---            ┆ t                ┆ nt               │
│ str        ┆ f64          ┆ f32           ┆ f64            ┆ ---              ┆ ---              │
│            ┆              ┆               ┆                ┆ f64              ┆ f64              │
╞════════════╪══════════════╪═══════════════╪════════════════╪══════════════════╪══════════════════╡
│ Garage     ┆ 0.613887     ┆ 0.108552      ┆ 0.505335       ┆ 0.333333         ┆ 0.30303          │
│ R&B        ┆ 0.516795     ┆ 0.104959      ┆ 0.411836       ┆ 0.069444         ┆ 0.007848         │
│ Breakbeat  ┆ 0.388593     ┆ 0.144539      ┆ 0.244054       ┆ 0.471429         ┆ 0.415385         │
│ Pop        ┆ 0.341279     ┆ 

## 6. Per-Label Analysis

Compare performance improvements for each label.

In [14]:
from models import evaluate_model, predict_with_threshold

# Evaluate with default 0.5 threshold
print("="*70)
print(f"TEST SET EVALUATION - {best_model_name} with Default Threshold (0.5)")
print("="*70)
default_test_metrics = evaluate_model(
    model=best_model,
    X_test=result.X_test,
    y_test=result.y_test,
    tags=result.tags,
    verbose=True
)

# Evaluate with optimized thresholds
print("\n" + "="*70)
print(f"TEST SET EVALUATION - {best_model_name} with Optimized Thresholds")
print("="*70)
_, test_probs = predict_with_threshold(best_model, result.X_test, threshold=0.5)
y_pred_optimized = (test_probs >= best_thresholds).astype(int)

# Calculate metrics manually for optimized thresholds
from sklearn.metrics import precision_recall_fscore_support, hamming_loss, accuracy_score

y_test_np = result.y_test.to_numpy()
hamming = hamming_loss(y_test_np, y_pred_optimized)
exact_match = accuracy_score(y_test_np, y_pred_optimized)
precision, recall, f1, support = precision_recall_fscore_support(
    y_test_np, y_pred_optimized, average=None, zero_division=0
)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test_np, y_pred_optimized, average="macro", zero_division=0
)

print(f"Hamming Loss: {hamming:.4f}")
print(f"Exact Match Accuracy: {exact_match:.4f}")
print(f"\nMacro F1: {macro_f1:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")

print(f"\n" + "="*70)
print("COMPARISON")
print("="*70)
print(f"Default threshold macro F1: {default_test_metrics['macro_f1']:.4f}")
print(f"Optimized threshold macro F1: {macro_f1:.4f}")
print(f"Improvement: {macro_f1 - default_test_metrics['macro_f1']:+.4f}")

TEST SET EVALUATION - XGBoost with Default Threshold (0.5)

MODEL EVALUATION RESULTS
Hamming Loss: 0.1405
  (Average fraction of labels incorrectly predicted per sample)

Exact Match Accuracy: 0.0600
  (Fraction of samples with ALL labels predicted correctly)

Macro-Averaged Metrics:
  (All labels weighted equally)
  Precision: 0.3261
  Recall:    0.3081
  F1-Score:  0.3052

Weighted-Averaged Metrics:
  (Labels weighted by number of samples)
  Precision: 0.4453
  Recall:    0.5000
  F1-Score:  0.4584

Per-Label Metrics (sorted by F1-score):
shape: (22, 5)
┌────────────┬───────────┬──────────┬──────────┬─────────┐
│ tag        ┆ precision ┆ recall   ┆ f1_score ┆ support │
│ ---        ┆ ---       ┆ ---      ┆ ---      ┆ ---     │
│ str        ┆ f64       ┆ f64      ┆ f64      ┆ i64     │
╞════════════╪═══════════╪══════════╪══════════╪═════════╡
│ Soul       ┆ 0.675676  ┆ 0.877193 ┆ 0.763359 ┆ 57      │
│ House      ┆ 0.642857  ┆ 0.923077 ┆ 0.757895 ┆ 39      │
│ Bass       ┆ 0.545455  

## 7. Save Both Models with Metadata

Save both Random Forest and XGBoost models with all their configuration, thresholds, and preprocessing components.

In [15]:
from models import save_model_comparison

# Prepare models to save
models_to_save = {
    "random_forest": {
        "model": rf_results['best_model'],
        "thresholds": rf_threshold_results['thresholds'],
        "hyperparams": rf_results['best_params'],
        "metrics": {
            "macro_f1": rf_threshold_results['optimized_metrics']['macro_f1'],
            "macro_precision": rf_threshold_results['optimized_metrics']['macro_precision'],
            "macro_recall": rf_threshold_results['optimized_metrics']['macro_recall'],
        }
    },
    "xgboost": {
        "model": xgb_results['best_model'],
        "thresholds": xgb_threshold_results['thresholds'],
        "hyperparams": xgb_results['best_params'],
        "metrics": {
            "macro_f1": xgb_threshold_results['optimized_metrics']['macro_f1'],
            "macro_precision": xgb_threshold_results['optimized_metrics']['macro_precision'],
            "macro_recall": xgb_threshold_results['optimized_metrics']['macro_recall'],
        }
    }
}

# Save all models at once
saved_paths = save_model_comparison(
    models_dict=models_to_save,
    tags=result.tags,
    save_dir="../models",
    tag_group="Genre",
    scaler=result.scaler,
    pca=result.pca
)

print("\nSaved model files:")
for model_name, paths in saved_paths.items():
    print(f"\n{model_name.upper()}:")
    for file_type, path in paths.items():
        if path:
            print(f"  {file_type}: {path}")


MODEL SAVED SUCCESSFULLY
Model name: random_forest_Genre
Tag group: Genre
Number of labels: 22

Files saved:
  Model:         ../models/random_forest_Genre_20251015_232027_model.pkl
  Configuration: ../models/random_forest_Genre_20251015_232027_config.json
  Thresholds:    ../models/random_forest_Genre_20251015_232027_thresholds.npy
  Scaler:        ../models/random_forest_Genre_20251015_232027_scaler.pkl

Metrics:
  macro_f1: 0.4909
  macro_precision: 0.4601
  macro_recall: 0.6430


MODEL SAVED SUCCESSFULLY
Model name: xgboost_Genre
Tag group: Genre
Number of labels: 22

Files saved:
  Model:         ../models/xgboost_Genre_20251015_232027_model.pkl
  Configuration: ../models/xgboost_Genre_20251015_232027_config.json
  Thresholds:    ../models/xgboost_Genre_20251015_232027_thresholds.npy
  Scaler:        ../models/xgboost_Genre_20251015_232027_scaler.pkl

Metrics:
  macro_f1: 0.5060
  macro_precision: 0.4732
  macro_recall: 0.6371


SAVED 2 MODELS


Saved model files:

RANDOM_FOREST: